# Plantilla de ejercicio — usa el machote

Copia este archivo para cada ejercicio nuevo y cambia lo que necesites.

---
## ¿Cómo importar el machote desde otra carpeta?

El problema es que Python solo busca imports en la carpeta actual del notebook.
Si el machote está en otra carpeta, hay que decirle explícitamente dónde buscarlo.

**Pasos:**
1. En VSCode, haz clic derecho sobre `machote_ML.py` → **Copy Path**
2. Pega esa ruta abajo, pero **quita el nombre del archivo** (solo la carpeta)
3. Cambia las diagonales invertidas `\\` por diagonales normales `/`

**Ejemplo:**
```
Si Copy Path te da:  C:\Users\Diana\DIPLOMADO-RNA\Machote_ML\machote_ML.py
Tú pones:           C:/Users/Diana/DIPLOMADO-RNA/Machote_ML
```

In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 1 — IMPORTAR EL MACHOTE
# Solo necesitas cambiar la ruta de abajo.
# Corre esta celda UNA VEZ al inicio de cada sesión.
# ════════════════════════════════════════════════════════════
import sys

# ↓ CAMBIA ESTA RUTA por la tuya (usa / no \)
RUTA_MACHOTE = r"C:/Users/Diana/DIPLOMADO-RNA/Machote_ML"

if RUTA_MACHOTE not in sys.path:
    sys.path.append(RUTA_MACHOTE)

from machote_ML import *

# Si sale error 'No module named machote_ML':
#   1. Verifica que la ruta apunte a la CARPETA (no al archivo .py)
#   2. Verifica que el archivo se llame exactamente machote_ML.py
#   3. Prueba correr: import os; print(os.listdir(RUTA_MACHOTE))
#      para ver qué archivos hay en esa carpeta

In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 2 — CARGAR DATOS
# Elige UNA de las tres opciones según de dónde vengan los datos
# ════════════════════════════════════════════════════════════
from sklearn.datasets import load_breast_cancer, load_iris, load_wine

# OPCIÓN A: desde sklearn (más fácil para practicar)
X, Y, clases = cargar_desde_sklearn(load_breast_cancer)

# OPCIÓN B: desde UCI (descomenta si la necesitas)
# X, Y = cargar_desde_uci(17, 'Diagnosis', {'M': 0, 'B': 1})
# clases = ['Maligno', 'Benigno']

# OPCIÓN C: desde CSV (descomenta si la necesitas)
# X, Y = cargar_desde_csv(r"C:/ruta/a/tu/archivo.csv", 'columna_resultado')
# clases = None

In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 3 — EXPLORAR (SIEMPRE ANTES DE MODELAR)
# Este bloque te da toda la información que necesitas para
# entender los datos antes de tomar decisiones.
# ════════════════════════════════════════════════════════════

# Paso 3a: panorama general
resumen_rapido(X, Y)

In [ ]:
# Paso 3b: correlación de cada columna con Y
# MIRA BIEN ESTA GRÁFICA antes de continuar:
#   - ¿Las barras son mayormente positivas o negativas?
#   - ¿Cuántas superan el umbral?
#   - La función te dirá si necesitas labels=[0,1] o [1,0]
correlaciones = ver_correlacion_con_y(X, Y, umbral=0.5)

In [ ]:
# Paso 3c: mapa de calor (opcional pero recomendado)
# Busca celdas con valor cercano a 1.0 FUERA de la diagonal principal.
# Esas son columnas redundantes entre sí.
ver_mapa_calor(X, Y, top_n=15)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 4 — SELECCIONAR CARACTERÍSTICAS
# ════════════════════════════════════════════════════════════
X_reducido, features = seleccionar_features(X, Y, umbral=0.5)

# Si quedan demasiadas columnas → sube umbral a 0.6 o 0.7
# Si quedan muy pocas (menos de 5) → baja umbral a 0.4
# No hay número mágico: experimenta

In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 5 — DIVIDIR EN TRAIN Y TEST
# ════════════════════════════════════════════════════════════
X_train, X_test, y_train, y_test = dividir_datos(
    X_reducido, Y,
    proporcion_test=0.25,  # 25% para test
    semilla=42,            # cualquier número, fija la aleatoriedad
    estratificar=True      # True para clasificación, False para regresión
)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 6 — PREPARAR DATOS
# Elige UNA opción según el modelo que vayas a usar:
#   MPNeuron         → binarizar_datos()
#   Cualquier otro   → escalar_datos()
# ════════════════════════════════════════════════════════════

# OPCIÓN A: para MPNeuron (pasa las correlaciones para detectar dirección)
# corr_reducido = correlaciones[features]  # solo las columnas que conservamos
# X_train_bin, X_test_bin = binarizar_datos(X_train, X_test, corr_reducido)

# OPCIÓN B: para cualquier otro modelo (regresión, SVM, redes neuronales)
X_train_sc, X_test_sc = escalar_datos(X_train, X_test, metodo='standard')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 7 — ENTRENAR EL MODELO
# Aquí pones el modelo que estés usando en clase.
# ════════════════════════════════════════════════════════════
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train_sc, y_train)
print("Modelo entrenado")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELDA 8 — PREDECIR Y EVALUAR
# ════════════════════════════════════════════════════════════
Y_pred = modelo.predict(X_test_sc)
acc = evaluar_clasificacion(y_test, Y_pred, clases)